In [5]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
import pygwalker as pyg

this_path = Path(__file__) if '__file__' in globals() else Path("<unknown>.ipynb").resolve()
work_path = next((p for p in this_path.parents if p.name == "research"), None)
tools_path = work_path / Path("../torch-tools")
sys.path.append(str(tools_path))

from run_manager import RunViewer

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
rv = RunViewer(exp_path=this_path.parent)
df_base = rv.fetch_results(met_listed=True)

nested_columns = [name for name, dtype in zip(df_base.columns, df_base.dtypes) if dtype.is_nested()]
df_base = df_base.with_columns([pl.col(name).list.last().alias(f"{name}") for name in nested_columns])

pyg.walk(df_base)

Box(children=(HTML(value='\n<div id="ifr-pyg-000636a8b649fe8dQxZ2jmnhW4XSc0Je" style="height: auto">\n    <hea…

In [ ]:
df = df_base

# df = df.filter(pl.col("fils").is_in([32, 16, 8, 4]))

piv_values = ["val_acc"] # 表示する値
piv_index = ["fils", "ensembles"] # 縦軸
piv_on = "max_lr" # 横軸

# agg = "len"
agg = "mean"

ext_column = "train_dataset" # このカラムの要素ごとにheatmapを表示

ext_l = df[ext_column].unique()
for ext in ext_l:
    # pivot table 作成
    df_ext = df.filter(pl.col(ext_column) == ext)
    df_piv = df_ext.pivot(values=piv_values, index=piv_index, on=piv_on, sort_columns=True, aggregate_function=agg)

    # カラムが文字列順になっているため、数字部分をソート
    _num_columns = sorted(int(x) for x in df_piv.columns if x.isdigit())
    new_columns = [str(_num_columns.pop(0)) if x.isdigit() else x for x in df_piv.columns]
    df_piv = df_piv.select(new_columns)

    # (fil, ensemble) の形式に
    df_piv = df_piv.with_columns(("(" + pl.col("fils").cast(pl.String) + "," + pl.col("ensembles").cast(pl.String)).alias("(fils, ensembles)") + ")")
    df_piv = df_piv.select(["(fils, ensembles)"] + new_columns).select(pl.exclude(["fils", "ensembles"]))

    # 0列目がx軸ラベル、1列目以降がy軸ラベルになる df を heat map に変換
    square_size = 0.75
    hm_x = df_piv.columns[1:]
    hm_y = df_piv[df_piv.columns[0]]
    data = df_piv.select(hm_x).to_numpy()
    annot = data.copy()

    # 正規化の方向を設定
    axis = 1    # 0: 行方向, 1: 列方向

    # min-max 正規化
    # min_vals = data.min(axis=axis, keepdims=True)
    # max_vals = data.max(axis=axis, keepdims=True)
    # data = (data - min_vals) / (max_vals - min_vals + 1e-8)  # ゼロ除算対策
    
    # Zスコア正規化
    mean_vals = data.mean(axis=axis, keepdims=True)
    std_vals = data.std(axis=axis, keepdims=True)
    data = (data - mean_vals) / (std_vals + 1e-8)  # ゼロ除算対策

    annot *= 100
    sp_kwargs = {}
    hm_kwargs = {"cmap": "Blues_r", "cbar": False, "fmt": ".1f"}
    # hm_kwargs = {"cmap": "Greens_r", "cbar": False, "fmt": ".1f"}

    # plot
    sns.set_theme()
    fig, ax = plt.subplots(figsize=(len(hm_x)*square_size, len(hm_y)*square_size), **sp_kwargs)
    ax = sns.heatmap(data, annot=annot, square=True, **hm_kwargs)

    # memo
    # xticklabels=pl.Series(df_piv_val.columns)
    # xticklabels=pl.Series(df_piv_val.columns).str.head(4)

    ax.set_title(f"{ext_column}: {ext}", fontsize=12)
    ax.set_xlabel(piv_on, fontsize=10)
    ax.set_ylabel(hm_y.name, fontsize=10, rotation=90)
    ax.set_xticklabels(hm_x, fontsize=10, rotation=0)
    ax.set_yticklabels(hm_y, fontsize=10, rotation=0)
    plt.show()


NameError: name 'df_base' is not defined

In [ ]:
x_col = "epoch"
y_cols = ["train_acc", "val_acc"]

fig, ax = plt.subplots()
# fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 5), squeeze=False)
for y_col in y_cols:
    x = run_csv[x_col].to_list()
    y = run_csv[y_col].to_list()
    # Noneの値を除外
    x, y = zip(*[(xi, yi) for xi, yi in zip(x, y) if yi is not None])
    ax.plot(x, y, label=y_col)
ax.legend()
ax.set_ylim([0, 1])
ax.set_title(f"10000data - SW", fontsize=12)
plt.show()


FileNotFoundError: No such file or directory (os error 2): /home/tat/research/ee/20250507_mn_compare/exp_opt/runs/137/_metrics.csv